In [1]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from utils.config import (
    SEED,
    NUM_CLASSES,
    PRETRAINED_MAX_LEN,
    PRETRAINED_LR,
    NUM_WORKERS,
    RESULTS_DIR,
    CONFUSION_DIR,
    MODEL_DIR,
    device,
    set_seed,
)

from utils.metrics import (
    calculate_metrics,
    get_classification_report,
    print_metrics,
)


# =========================================================
# 0. Experiment Configuration
# =========================================================

MODEL_NAME = "distilbert/distilbert-base-uncased"

BATCH_SIZE = 16

GRADIENT_ACCUMULATION_STEPS = 4

EPOCHS = 3

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.1

NUM_LABELS = NUM_CLASSES

MAX_LENGTH = PRETRAINED_MAX_LEN

LEARNING_RATE = PRETRAINED_LR


# =========================================================
# 1. Dataset
# =========================================================

class EmotionDataset(Dataset):

    def __init__(
        self,
        dataframe
    ):
        self.texts = (
            dataframe["text"]
            .astype(str)
            .tolist()
        )

        self.labels = (
            dataframe["label"]
            .astype(int)
            .tolist()
        )

    def __len__(self):

        return len(self.texts)

    def __getitem__(
        self,
        index
    ):

        return {
            "text": self.texts[index],
            "labels": self.labels[index],
        }


# =========================================================
# 2. Dynamic Padding Collator
# =========================================================

class DistilBERTCollator:

    def __init__(
        self,
        tokenizer,
        max_length
    ):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(
        self,
        batch
    ):

        texts = [
            item["text"]
            for item in batch
        ]

        labels = [
            item["labels"]
            for item in batch
        ]


        # -------------------------------------------------
        # Tokenization
        # -------------------------------------------------
        #
        # padding=True
        # → 현재 batch에서 가장 긴 sequence에 맞춰 padding
        #
        # truncation=True
        # → MAX_LENGTH를 초과하는 sequence 잘라냄
        #
        # -------------------------------------------------

        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )


        encoded["labels"] = torch.tensor(
            labels,
            dtype=torch.long
        )

        return encoded


# =========================================================
# 3. Count Parameters
# =========================================================

def count_parameters(model):

    total_params = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_params = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    return (
        total_params,
        trainable_params
    )


# =========================================================
# 4. Move Batch to Device
# =========================================================

def move_batch_to_device(
    batch
):

    return {
        key: value.to(device)
        for key, value in batch.items()
    }


# =========================================================
# 5. Train One Epoch
# =========================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    scaler
):

    model.train()

    total_loss = 0.0

    total_samples = 0

    optimizer.zero_grad(
        set_to_none=True
    )


    for step, batch in enumerate(
        dataloader
    ):

        batch = move_batch_to_device(
            batch
        )


        # -------------------------------------------------
        # Mixed Precision
        # -------------------------------------------------

        use_amp = (
            device.type == "cuda"
        )


        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp
        ):

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )

            loss = outputs.loss


        # -------------------------------------------------
        # Gradient Accumulation
        # -------------------------------------------------

        loss_for_backward = (
            loss
            / GRADIENT_ACCUMULATION_STEPS
        )


        if use_amp:

            scaler.scale(
                loss_for_backward
            ).backward()

        else:

            loss_for_backward.backward()


        batch_size = (
            batch["labels"].size(0)
        )

        total_loss += (
            loss.item()
            * batch_size
        )

        total_samples += batch_size


        # -------------------------------------------------
        # Optimizer Step
        # -------------------------------------------------

        is_accumulation_step = (
            (step + 1)
            % GRADIENT_ACCUMULATION_STEPS
            == 0
        )

        is_last_step = (
            step + 1
            == len(dataloader)
        )


        if (
            is_accumulation_step
            or is_last_step
        ):

            if use_amp:

                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

            else:

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                optimizer.step()


            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )


    average_loss = (
        total_loss
        / total_samples
    )

    return average_loss


# =========================================================
# 6. Evaluation
# =========================================================

def evaluate(
    model,
    dataloader
):

    model.eval()

    total_loss = 0.0

    total_samples = 0

    all_labels = []

    all_predictions = []


    with torch.no_grad():

        for batch in dataloader:

            batch = move_batch_to_device(
                batch
            )


            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )


            loss = outputs.loss

            logits = outputs.logits


            batch_size = (
                batch["labels"].size(0)
            )


            total_loss += (
                loss.item()
                * batch_size
            )

            total_samples += batch_size


            predictions = torch.argmax(
                logits,
                dim=1
            )


            all_labels.extend(
                batch["labels"]
                .detach()
                .cpu()
                .numpy()
            )

            all_predictions.extend(
                predictions
                .detach()
                .cpu()
                .numpy()
            )


    average_loss = (
        total_loss
        / total_samples
    )


    metrics = calculate_metrics(
        all_labels,
        all_predictions
    )


    return (
        average_loss,
        all_labels,
        all_predictions,
        metrics,
    )


# =========================================================
# 7. Save Training Curve
# =========================================================

def save_training_curve(
    history,
    save_path
):

    epochs = range(
        1,
        len(history["train_loss"]) + 1
    )


    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        epochs,
        history["train_loss"],
        marker="o",
        label="Train Loss"
    )

    plt.plot(
        epochs,
        history["val_loss"],
        marker="o",
        label="Validation Loss"
    )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "Loss"
    )

    plt.title(
        "DistilBERT Training Curve"
    )

    plt.xticks(
        list(epochs)
    )

    plt.legend()

    plt.grid(
        alpha=0.3
    )

    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=150
    )

    plt.close()


# =========================================================
# 8. Save Macro F1 Curve
# =========================================================

def save_f1_curve(
    history,
    save_path
):

    epochs = range(
        1,
        len(history["val_macro_f1"]) + 1
    )


    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        epochs,
        history["val_macro_f1"],
        marker="o",
        label="Validation Macro F1"
    )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "Macro F1"
    )

    plt.title(
        "DistilBERT Validation Macro F1"
    )

    plt.xticks(
        list(epochs)
    )

    plt.legend()

    plt.grid(
        alpha=0.3
    )

    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=150
    )

    plt.close()


# =========================================================
# 9. Save Confusion Matrix
# =========================================================

def save_confusion_matrix(
    y_true,
    y_pred,
    save_path
):

    cm = confusion_matrix(
        y_true,
        y_pred
    )


    display = ConfusionMatrixDisplay(
        confusion_matrix=cm
    )


    display.plot(
        values_format="d"
    )


    plt.title(
        "DistilBERT Confusion Matrix"
    )

    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=150
    )

    plt.close()


# =========================================================
# 10. Main
# =========================================================

def main():

    # =====================================================
    # Seed
    # =====================================================

    set_seed(SEED)


    print("=" * 70)
    print("06-1. DistilBERT Fine-tuning")
    print("=" * 70)


    # =====================================================
    # Experiment Configuration
    # =====================================================

    print(
        f"Device                 : {device}"
    )

    print(
        f"Model                  : {MODEL_NAME}"
    )

    print(
        f"Max Length             : {MAX_LENGTH}"
    )

    print(
        f"Batch Size             : {BATCH_SIZE}"
    )

    print(
        f"Gradient Accumulation  : "
        f"{GRADIENT_ACCUMULATION_STEPS}"
    )

    print(
        f"Effective Batch Size   : "
        f"{BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}"
    )

    print(
        f"Epochs                 : {EPOCHS}"
    )

    print(
        f"Learning Rate          : {LEARNING_RATE}"
    )

    print(
        f"Weight Decay           : {WEIGHT_DECAY}"
    )

    print(
        f"Warmup Ratio           : {WARMUP_RATIO}"
    )

    print()


    # =====================================================
    # 1. Load Splits
    # =====================================================

    print("Dataset")
    print("-" * 70)


    train_df = pd.read_csv(
        "results/splits/train.csv"
    )

    val_df = pd.read_csv(
        "results/splits/validation.csv"
    )

    test_df = pd.read_csv(
        "results/splits/test.csv"
    )


    print(
        f"Train      : {len(train_df):,}"
    )

    print(
        f"Validation : {len(val_df):,}"
    )

    print(
        f"Test       : {len(test_df):,}"
    )

    print()


    # =====================================================
    # 2. Tokenizer
    # =====================================================

    print("Tokenizer")
    print("-" * 70)


    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )


    print(
        f"Tokenizer  : "
        f"{tokenizer.__class__.__name__}"
    )

    print(
        f"Vocab Size : "
        f"{tokenizer.vocab_size:,}"
    )

    print()


    # =====================================================
    # 3. Dataset
    # =====================================================

    train_dataset = EmotionDataset(
        train_df
    )

    val_dataset = EmotionDataset(
        val_df
    )

    test_dataset = EmotionDataset(
        test_df
    )


    # =====================================================
    # 4. Collator
    # =====================================================

    collator = DistilBERTCollator(
        tokenizer=tokenizer,
        max_length=MAX_LENGTH
    )


    # =====================================================
    # 5. DataLoader
    # =====================================================

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=(
            device.type == "cuda"
        ),
    )


    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=(
            device.type == "cuda"
        ),
    )


    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=(
            device.type == "cuda"
        ),
    )


    # =====================================================
    # 6. Check Tokenization
    # =====================================================

    print("Tokenization Check")
    print("-" * 70)


    sample_text = (
        train_df.iloc[0]["text"]
    )

    sample_label = (
        train_df.iloc[0]["label"]
    )


    sample_encoding = tokenizer(
        sample_text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )


    print(
        f"Text  : {sample_text}"
    )

    print(
        f"Label : {sample_label}"
    )

    print(
        f"Input IDs Shape : "
        f"{sample_encoding['input_ids'].shape}"
    )

    print(
        f"Attention Mask Shape : "
        f"{sample_encoding['attention_mask'].shape}"
    )

    print(
        f"First Token ID : "
        f"{sample_encoding['input_ids'][0, 0].item()}"
    )

    print()


    # =====================================================
    # 7. Model
    # =====================================================

    print("Model")
    print("-" * 70)


    id2label = {
        0: "LABEL_0",
        1: "LABEL_1",
        2: "LABEL_2",
        3: "LABEL_3",
        4: "LABEL_4",
        5: "LABEL_5",
    }


    label2id = {
        "LABEL_0": 0,
        "LABEL_1": 1,
        "LABEL_2": 2,
        "LABEL_3": 3,
        "LABEL_4": 4,
        "LABEL_5": 5,
    }


    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
    )


    model = model.to(device)


    total_params, trainable_params = (
        count_parameters(model)
    )


    print(
        f"Total Parameters     : "
        f"{total_params:,}"
    )

    print(
        f"Trainable Parameters : "
        f"{trainable_params:,}"
    )

    print()


    # =====================================================
    # 8. Optimizer
    # =====================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )


    # =====================================================
    # 9. Scheduler
    # =====================================================

    total_update_steps = (
        (
            len(train_loader)
            + GRADIENT_ACCUMULATION_STEPS
            - 1
        )
        // GRADIENT_ACCUMULATION_STEPS
    ) * EPOCHS


    warmup_steps = int(
        total_update_steps
        * WARMUP_RATIO
    )


    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_update_steps,
    )


    print("Training Schedule")
    print("-" * 70)

    print(
        f"Train Batches         : "
        f"{len(train_loader):,}"
    )

    print(
        f"Optimizer Steps/Epoch : "
        f"{(
            len(train_loader)
            + GRADIENT_ACCUMULATION_STEPS
            - 1
        ) // GRADIENT_ACCUMULATION_STEPS:,}"
    )

    print(
        f"Total Update Steps    : "
        f"{total_update_steps:,}"
    )

    print(
        f"Warmup Steps          : "
        f"{warmup_steps:,}"
    )

    print()


    # =====================================================
    # 10. Training
    # =====================================================

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_macro_f1": [],
    }


    best_macro_f1 = -1.0

    best_epoch = -1

    best_model_path = (
        MODEL_DIR
        / "distilbert_best.pt"
    )


    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(
            device.type == "cuda"
        )
    )


    training_start = time.perf_counter()


    for epoch in range(
        1,
        EPOCHS + 1
    ):

        epoch_start = time.perf_counter()


        print("=" * 70)

        print(
            f"Epoch [{epoch:02d}/{EPOCHS}]"
        )

        print("=" * 70)


        train_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
        )


        (
            val_loss,
            val_true,
            val_pred,
            val_metrics,
        ) = evaluate(
            model=model,
            dataloader=val_loader,
        )


        val_macro_f1 = (
            val_metrics["macro_f1"]
        )


        history["train_loss"].append(
            train_loss
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_macro_f1"].append(
            val_macro_f1
        )


        epoch_time = (
            time.perf_counter()
            - epoch_start
        )


        print(
            f"Train Loss       : "
            f"{train_loss:.4f}"
        )

        print(
            f"Val Loss         : "
            f"{val_loss:.4f}"
        )

        print(
            f"Val Accuracy     : "
            f"{val_metrics['accuracy']:.4f}"
        )

        print(
            f"Val Macro F1     : "
            f"{val_macro_f1:.4f}"
        )

        print(
            f"Epoch Time       : "
            f"{epoch_time:.2f}s"
        )


        # -------------------------------------------------
        # Best Model
        # -------------------------------------------------

        if val_macro_f1 > best_macro_f1:

            best_macro_f1 = (
                val_macro_f1
            )

            best_epoch = epoch


            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": (
                        model.state_dict()
                    ),
                    "optimizer_state_dict": (
                        optimizer.state_dict()
                    ),
                    "scheduler_state_dict": (
                        scheduler.state_dict()
                    ),
                    "best_macro_f1": (
                        best_macro_f1
                    ),
                    "model_name": (
                        MODEL_NAME
                    ),
                    "max_length": (
                        MAX_LENGTH
                    ),
                    "learning_rate": (
                        LEARNING_RATE
                    ),
                    "batch_size": (
                        BATCH_SIZE
                    ),
                    "gradient_accumulation_steps": (
                        GRADIENT_ACCUMULATION_STEPS
                    ),
                },
                best_model_path
            )


            print(
                f"Best model saved "
                f"(Val Macro F1 = "
                f"{best_macro_f1:.4f})"
            )


        print()


    total_training_time = (
        time.perf_counter()
        - training_start
    )


    # =====================================================
    # 11. Save Curves
    # =====================================================

    save_training_curve(
        history,
        RESULTS_DIR
        / "distilbert_training_curve.png"
    )


    save_f1_curve(
        history,
        RESULTS_DIR
        / "distilbert_val_macro_f1.png"
    )


    # =====================================================
    # 12. Load Best Model
    # =====================================================

    print("=" * 70)
    print("Best Model")
    print("=" * 70)


    checkpoint = torch.load(
        best_model_path,
        map_location=device,
        weights_only=False
    )


    model.load_state_dict(
        checkpoint["model_state_dict"]
    )


    print(
        f"Best Epoch          : "
        f"{checkpoint['epoch']}"
    )

    print(
        f"Best Val Macro F1    : "
        f"{checkpoint['best_macro_f1']:.4f}"
    )

    print()


    # =====================================================
    # 13. Final Test Evaluation
    # =====================================================

    print("=" * 70)
    print("Final Test Evaluation")
    print("=" * 70)


    inference_start = time.perf_counter()


    (
        test_loss,
        test_true,
        test_pred,
        test_metrics,
    ) = evaluate(
        model=model,
        dataloader=test_loader,
    )


    inference_time = (
        time.perf_counter()
        - inference_start
    )


    print()

    print(
        f"Test Loss         : "
        f"{test_loss:.4f}"
    )


    print_metrics(
        test_metrics
    )


    print()

    print(
        f"Inference Time    : "
        f"{inference_time:.2f}s"
    )

    print()


    # =====================================================
    # 14. Classification Report
    # =====================================================

    print("Classification Report")
    print("-" * 70)

    print(
        get_classification_report(
            test_true,
            test_pred
        )
    )


    # =====================================================
    # 15. Confusion Matrix
    # =====================================================

    confusion_path = (
        CONFUSION_DIR
        / "distilbert.png"
    )


    save_confusion_matrix(
        test_true,
        test_pred,
        confusion_path
    )


    # =====================================================
    # 16. Save Tokenizer
    # =====================================================

    tokenizer_path = (
        MODEL_DIR
        / "distilbert_tokenizer"
    )


    tokenizer.save_pretrained(
        tokenizer_path
    )


    # =====================================================
    # 17. Save Model in Hugging Face Format
    # =====================================================

    model_path = (
        MODEL_DIR
        / "distilbert_model"
    )


    model.save_pretrained(
        model_path
    )


    # =====================================================
    # 18. Save Metrics
    # =====================================================

    metrics_record = {
        "model": "DistilBERT",
        "model_name": MODEL_NAME,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_macro_f1,
        "test_loss": test_loss,
        "accuracy": test_metrics["accuracy"],
        "macro_precision": (
            test_metrics["macro_precision"]
        ),
        "macro_recall": (
            test_metrics["macro_recall"]
        ),
        "macro_f1": (
            test_metrics["macro_f1"]
        ),
        "weighted_precision": (
            test_metrics["weighted_precision"]
        ),
        "weighted_recall": (
            test_metrics["weighted_recall"]
        ),
        "weighted_f1": (
            test_metrics["weighted_f1"]
        ),
        "training_time_sec": (
            total_training_time
        ),
        "inference_time_sec": (
            inference_time
        ),
        "total_parameters": (
            total_params
        ),
        "trainable_parameters": (
            trainable_params
        ),
        "max_length": MAX_LENGTH,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": (
            GRADIENT_ACCUMULATION_STEPS
        ),
        "epochs": EPOCHS,
    }


    metrics_path = (
        RESULTS_DIR
        / "distilbert_metrics.csv"
    )


    pd.DataFrame(
        [metrics_record]
    ).to_csv(
        metrics_path,
        index=False
    )


    # =====================================================
    # 19. Result Summary
    # =====================================================

    print("=" * 70)
    print("Saved Artifacts")
    print("=" * 70)

    print(
        f"Checkpoint : "
        f"{best_model_path}"
    )

    print(
        f"Model      : "
        f"{model_path}"
    )

    print(
        f"Tokenizer  : "
        f"{tokenizer_path}"
    )

    print(
        f"Curve      : "
        f"{RESULTS_DIR / 'distilbert_training_curve.png'}"
    )

    print(
        f"F1 Curve   : "
        f"{RESULTS_DIR / 'distilbert_val_macro_f1.png'}"
    )

    print(
        f"Confusion  : "
        f"{confusion_path}"
    )

    print(
        f"Metrics    : "
        f"{metrics_path}"
    )

    print()


    print("=" * 70)
    print("06-1. DistilBERT Fine-tuning Completed")
    print("=" * 70)


# =========================================================
# Entry Point
# =========================================================

if __name__ == "__main__":

    main()

06-1. DistilBERT Fine-tuning
Device                 : cuda
Model                  : distilbert/distilbert-base-uncased
Max Length             : 128
Batch Size             : 16
Gradient Accumulation  : 4
Effective Batch Size   : 64
Epochs                 : 3
Learning Rate          : 2e-05
Weight Decay           : 0.01
Warmup Ratio           : 0.1

Dataset
----------------------------------------------------------------------
Train      : 305,345
Validation : 65,422
Test       : 65,355

Tokenizer
----------------------------------------------------------------------
Tokenizer  : BertTokenizer
Vocab Size : 30,522

Tokenization Check
----------------------------------------------------------------------
Text  : im feeling rather rotten so im not very ambitious right now
Label : 0
Input IDs Shape : torch.Size([1, 13])
Attention Mask Shape : torch.Size([1, 13])
First Token ID : 101

Model
----------------------------------------------------------------------


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total Parameters     : 66,958,086
Trainable Parameters : 66,958,086

Training Schedule
----------------------------------------------------------------------
Train Batches         : 19,085
Optimizer Steps/Epoch : 4,772
Total Update Steps    : 14,316
Warmup Steps          : 1,431

Epoch [01/3]
Train Loss       : 0.2592
Val Loss         : 0.0898
Val Accuracy     : 0.9422
Val Macro F1     : 0.9053
Epoch Time       : 310.69s
Best model saved (Val Macro F1 = 0.9053)

Epoch [02/3]
Train Loss       : 0.0886
Val Loss         : 0.0878
Val Accuracy     : 0.9426
Val Macro F1     : 0.9053
Epoch Time       : 316.88s
Best model saved (Val Macro F1 = 0.9053)

Epoch [03/3]
Train Loss       : 0.0816
Val Loss         : 0.0893
Val Accuracy     : 0.9427
Val Macro F1     : 0.9040
Epoch Time       : 317.20s

Best Model
Best Epoch          : 2
Best Val Macro F1    : 0.9053

Final Test Evaluation

Test Loss         : 0.0882
Accuracy          : 0.9425
Macro Precision   : 0.9350
Macro Recall      : 0.8870
Macro

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Artifacts
Checkpoint : results/checkpoints/distilbert_best.pt
Model      : results/checkpoints/distilbert_model
Tokenizer  : results/checkpoints/distilbert_tokenizer
Curve      : results/distilbert_training_curve.png
F1 Curve   : results/distilbert_val_macro_f1.png
Confusion  : results/confusion_matrices/distilbert.png
Metrics    : results/distilbert_metrics.csv

06-1. DistilBERT Fine-tuning Completed


In [2]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from utils.config import (
    SEED,
    NUM_CLASSES,
    PRETRAINED_MAX_LEN,
    PRETRAINED_LR,
    NUM_WORKERS,
    RESULTS_DIR,
    CONFUSION_DIR,
    MODEL_DIR,
    device,
    set_seed,
)

from utils.metrics import (
    calculate_metrics,
    get_classification_report,
    print_metrics,
)


# ============================================================
# 06-2. BERT Baseline Fine-tuning
# ============================================================

MODEL_NAME = "google-bert/bert-base-uncased"

BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 4

EPOCHS = 3

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

NUM_LABELS = NUM_CLASSES
MAX_LENGTH = PRETRAINED_MAX_LEN
LEARNING_RATE = PRETRAINED_LR


# ============================================================
# Dataset
# ============================================================

class EmotionDataset(Dataset):

    def __init__(self, dataframe):

        self.texts = (
            dataframe["text"]
            .astype(str)
            .tolist()
        )

        self.labels = (
            dataframe["label"]
            .astype(int)
            .tolist()
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        return {
            "text": self.texts[index],
            "labels": self.labels[index],
        }


# ============================================================
# Dynamic Padding Collator
# ============================================================

class BERTCollator:

    def __init__(
        self,
        tokenizer,
        max_length
    ):

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):

        texts = [
            item["text"]
            for item in batch
        ]

        labels = [
            item["labels"]
            for item in batch
        ]

        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        encoded["labels"] = torch.tensor(
            labels,
            dtype=torch.long
        )

        return encoded


# ============================================================
# Parameter Count
# ============================================================

def count_parameters(model):

    total_params = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_params = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    return total_params, trainable_params


# ============================================================
# Move Batch
# ============================================================

def move_batch_to_device(batch):

    return {
        key: value.to(device)
        for key, value in batch.items()
    }


# ============================================================
# Train One Epoch
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    scaler,
):

    model.train()

    total_loss = 0.0
    total_samples = 0

    optimizer.zero_grad(
        set_to_none=True
    )

    for step, batch in enumerate(
        dataloader
    ):

        batch = move_batch_to_device(
            batch
        )

        use_amp = (
            device.type == "cuda"
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch[
                    "attention_mask"
                ],
                labels=batch["labels"],
            )

            loss = outputs.loss

        # ----------------------------------------------------
        # Gradient Accumulation
        # ----------------------------------------------------

        loss_for_backward = (
            loss
            / GRADIENT_ACCUMULATION_STEPS
        )

        if use_amp:

            scaler.scale(
                loss_for_backward
            ).backward()

        else:

            loss_for_backward.backward()

        batch_size = (
            batch["labels"].size(0)
        )

        total_loss += (
            loss.item()
            * batch_size
        )

        total_samples += batch_size

        # ----------------------------------------------------
        # Optimizer Step
        # ----------------------------------------------------

        is_accumulation_step = (
            (step + 1)
            % GRADIENT_ACCUMULATION_STEPS
            == 0
        )

        is_last_step = (
            step + 1
            == len(dataloader)
        )

        if (
            is_accumulation_step
            or is_last_step
        ):

            if use_amp:

                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

            else:

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                )

                optimizer.step()

            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

    return (
        total_loss
        / total_samples
    )


# ============================================================
# Evaluation
# ============================================================

def evaluate(
    model,
    dataloader,
):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for batch in dataloader:

            batch = move_batch_to_device(
                batch
            )

            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch[
                    "attention_mask"
                ],
                labels=batch["labels"],
            )

            loss = outputs.loss
            logits = outputs.logits

            batch_size = (
                batch["labels"].size(0)
            )

            total_loss += (
                loss.item()
                * batch_size
            )

            total_samples += batch_size

            predictions = torch.argmax(
                logits,
                dim=1
            )

            all_labels.extend(
                batch["labels"]
                .detach()
                .cpu()
                .numpy()
            )

            all_predictions.extend(
                predictions
                .detach()
                .cpu()
                .numpy()
            )

    average_loss = (
        total_loss
        / total_samples
    )

    metrics = calculate_metrics(
        all_labels,
        all_predictions
    )

    return (
        average_loss,
        all_labels,
        all_predictions,
        metrics,
    )


# ============================================================
# Training Curve
# ============================================================

def save_training_curve(
    history,
    save_path,
):

    epochs = range(
        1,
        len(history["train_loss"]) + 1
    )

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        epochs,
        history["train_loss"],
        marker="o",
        label="Train Loss",
    )

    plt.plot(
        epochs,
        history["val_loss"],
        marker="o",
        label="Validation Loss",
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")

    plt.title(
        "BERT Training Curve"
    )

    plt.xticks(
        list(epochs)
    )

    plt.legend()

    plt.grid(
        alpha=0.3
    )

    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=150
    )

    plt.close()


# ============================================================
# Validation Macro F1 Curve
# ============================================================

def save_f1_curve(
    history,
    save_path,
):

    epochs = range(
        1,
        len(history["val_macro_f1"]) + 1
    )

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        epochs,
        history["val_macro_f1"],
        marker="o",
        label="Validation Macro F1",
    )

    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")

    plt.title(
        "BERT Validation Macro F1"
    )

    plt.xticks(
        list(epochs)
    )

    plt.legend()

    plt.grid(
        alpha=0.3
    )

    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=150
    )

    plt.close()


# ============================================================
# Confusion Matrix
# ============================================================

def save_confusion_matrix(
    y_true,
    y_pred,
    save_path,
):

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm
    )

    display.plot(
        values_format="d"
    )

    plt.title(
        "BERT Confusion Matrix"
    )

    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=150
    )

    plt.close()


# ============================================================
# Main
# ============================================================

def main():

    set_seed(SEED)

    print("=" * 70)
    print(
        "06-2. BERT Fine-tuning"
    )
    print("=" * 70)

    print(
        f"Device                 : "
        f"{device}"
    )

    print(
        f"Model                  : "
        f"{MODEL_NAME}"
    )

    print(
        f"Max Length             : "
        f"{MAX_LENGTH}"
    )

    print(
        f"Batch Size             : "
        f"{BATCH_SIZE}"
    )

    print(
        f"Gradient Accumulation  : "
        f"{GRADIENT_ACCUMULATION_STEPS}"
    )

    print(
        f"Effective Batch Size   : "
        f"{BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}"
    )

    print(
        f"Epochs                 : "
        f"{EPOCHS}"
    )

    print(
        f"Learning Rate          : "
        f"{LEARNING_RATE}"
    )

    print(
        f"Weight Decay           : "
        f"{WEIGHT_DECAY}"
    )

    print(
        f"Warmup Ratio           : "
        f"{WARMUP_RATIO}"
    )

    print()


    # ========================================================
    # Dataset
    # ========================================================

    print("Dataset")
    print("-" * 70)

    train_df = pd.read_csv(
        "results/splits/train.csv"
    )

    val_df = pd.read_csv(
        "results/splits/validation.csv"
    )

    test_df = pd.read_csv(
        "results/splits/test.csv"
    )

    print(
        f"Train      : "
        f"{len(train_df):,}"
    )

    print(
        f"Validation : "
        f"{len(val_df):,}"
    )

    print(
        f"Test       : "
        f"{len(test_df):,}"
    )

    print()


    # ========================================================
    # Tokenizer
    # ========================================================

    print("Tokenizer")
    print("-" * 70)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    print(
        f"Tokenizer  : "
        f"{tokenizer.__class__.__name__}"
    )

    print(
        f"Vocab Size : "
        f"{tokenizer.vocab_size:,}"
    )

    print()


    # ========================================================
    # Dataset / DataLoader
    # ========================================================

    train_dataset = EmotionDataset(
        train_df
    )

    val_dataset = EmotionDataset(
        val_df
    )

    test_dataset = EmotionDataset(
        test_df
    )

    collator = BERTCollator(
        tokenizer=tokenizer,
        max_length=MAX_LENGTH
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=(
            device.type == "cuda"
        ),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=(
            device.type == "cuda"
        ),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=(
            device.type == "cuda"
        ),
    )


    # ========================================================
    # Tokenization Check
    # ========================================================

    print(
        "Tokenization Check"
    )

    print("-" * 70)

    sample_text = (
        train_df.iloc[0]["text"]
    )

    sample_label = (
        train_df.iloc[0]["label"]
    )

    sample_encoding = tokenizer(
        sample_text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    print(
        f"Text  : "
        f"{sample_text}"
    )

    print(
        f"Label : "
        f"{sample_label}"
    )

    print(
        f"Input IDs Shape : "
        f"{sample_encoding['input_ids'].shape}"
    )

    print(
        f"Attention Mask Shape : "
        f"{sample_encoding['attention_mask'].shape}"
    )

    print(
        f"First Token ID : "
        f"{sample_encoding['input_ids'][0, 0].item()}"
    )

    print()


    # ========================================================
    # Model
    # ========================================================

    print("Model")
    print("-" * 70)

    id2label = {
        0: "LABEL_0",
        1: "LABEL_1",
        2: "LABEL_2",
        3: "LABEL_3",
        4: "LABEL_4",
        5: "LABEL_5",
    }

    label2id = {
        "LABEL_0": 0,
        "LABEL_1": 1,
        "LABEL_2": 2,
        "LABEL_3": 3,
        "LABEL_4": 4,
        "LABEL_5": 5,
    }

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_NAME,
            num_labels=NUM_LABELS,
            id2label=id2label,
            label2id=label2id,
        )
    )

    model = model.to(device)

    total_params, trainable_params = (
        count_parameters(model)
    )

    print(
        f"Total Parameters     : "
        f"{total_params:,}"
    )

    print(
        f"Trainable Parameters : "
        f"{trainable_params:,}"
    )

    print()


    # ========================================================
    # Optimizer
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )


    # ========================================================
    # Scheduler
    # ========================================================

    total_update_steps = (
        (
            len(train_loader)
            + GRADIENT_ACCUMULATION_STEPS
            - 1
        )
        // GRADIENT_ACCUMULATION_STEPS
    ) * EPOCHS

    warmup_steps = int(
        total_update_steps
        * WARMUP_RATIO
    )

    scheduler = (
        get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=(
                total_update_steps
            ),
        )
    )


    # ========================================================
    # Training Schedule
    # ========================================================

    print(
        "Training Schedule"
    )

    print("-" * 70)

    print(
        f"Train Batches         : "
        f"{len(train_loader):,}"
    )

    optimizer_steps_per_epoch = (
        (
            len(train_loader)
            + GRADIENT_ACCUMULATION_STEPS
            - 1
        )
        // GRADIENT_ACCUMULATION_STEPS
    )

    print(
        f"Optimizer Steps/Epoch : "
        f"{optimizer_steps_per_epoch:,}"
    )

    print(
        f"Total Update Steps    : "
        f"{total_update_steps:,}"
    )

    print(
        f"Warmup Steps          : "
        f"{warmup_steps:,}"
    )

    print()


    # ========================================================
    # Training
    # ========================================================

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_macro_f1": [],
    }

    best_macro_f1 = -1.0
    best_epoch = -1

    best_model_path = (
        MODEL_DIR
        / "bert_best.pt"
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(
            device.type == "cuda"
        ),
    )

    training_start = (
        time.perf_counter()
    )


    for epoch in range(
        1,
        EPOCHS + 1
    ):

        epoch_start = (
            time.perf_counter()
        )

        print("=" * 70)

        print(
            f"Epoch "
            f"[{epoch:02d}/{EPOCHS}]"
        )

        print("=" * 70)


        train_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
        )


        (
            val_loss,
            val_true,
            val_pred,
            val_metrics,
        ) = evaluate(
            model=model,
            dataloader=val_loader,
        )


        val_macro_f1 = (
            val_metrics["macro_f1"]
        )


        history[
            "train_loss"
        ].append(
            train_loss
        )

        history[
            "val_loss"
        ].append(
            val_loss
        )

        history[
            "val_macro_f1"
        ].append(
            val_macro_f1
        )


        epoch_time = (
            time.perf_counter()
            - epoch_start
        )


        print(
            f"Train Loss       : "
            f"{train_loss:.4f}"
        )

        print(
            f"Val Loss         : "
            f"{val_loss:.4f}"
        )

        print(
            f"Val Accuracy     : "
            f"{val_metrics['accuracy']:.4f}"
        )

        print(
            f"Val Macro F1     : "
            f"{val_macro_f1:.4f}"
        )

        print(
            f"Epoch Time       : "
            f"{epoch_time:.2f}s"
        )


        # ====================================================
        # Best Model Selection
        # ====================================================

        if (
            val_macro_f1
            > best_macro_f1
        ):

            best_macro_f1 = (
                val_macro_f1
            )

            best_epoch = epoch


            torch.save(
                {
                    "epoch": epoch,

                    "model_state_dict":
                        model.state_dict(),

                    "optimizer_state_dict":
                        optimizer.state_dict(),

                    "scheduler_state_dict":
                        scheduler.state_dict(),

                    "best_macro_f1":
                        best_macro_f1,

                    "model_name":
                        MODEL_NAME,

                    "max_length":
                        MAX_LENGTH,

                    "learning_rate":
                        LEARNING_RATE,

                    "batch_size":
                        BATCH_SIZE,

                    "gradient_accumulation_steps":
                        GRADIENT_ACCUMULATION_STEPS,
                },

                best_model_path,
            )


            print(
                f"Best model saved "
                f"(Val Macro F1 = "
                f"{best_macro_f1:.4f})"
            )

        print()


    total_training_time = (
        time.perf_counter()
        - training_start
    )


    # ========================================================
    # Save Training Curves
    # ========================================================

    save_training_curve(
        history,
        RESULTS_DIR
        / "bert_training_curve.png",
    )

    save_f1_curve(
        history,
        RESULTS_DIR
        / "bert_val_macro_f1.png",
    )


    # ========================================================
    # Load Best Model
    # ========================================================

    print("=" * 70)
    print("Best Model")
    print("=" * 70)

    checkpoint = torch.load(
        best_model_path,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    print(
        f"Best Epoch          : "
        f"{checkpoint['epoch']}"
    )

    print(
        f"Best Val Macro F1    : "
        f"{checkpoint['best_macro_f1']:.4f}"
    )

    print()


    # ========================================================
    # Final Test
    # ========================================================

    print("=" * 70)
    print("Final Test Evaluation")
    print("=" * 70)


    inference_start = (
        time.perf_counter()
    )


    (
        test_loss,
        test_true,
        test_pred,
        test_metrics,
    ) = evaluate(
        model=model,
        dataloader=test_loader,
    )


    inference_time = (
        time.perf_counter()
        - inference_start
    )


    print()

    print(
        f"Test Loss         : "
        f"{test_loss:.4f}"
    )

    print_metrics(
        test_metrics
    )

    print()

    print(
        f"Inference Time    : "
        f"{inference_time:.2f}s"
    )

    print()


    # ========================================================
    # Classification Report
    # ========================================================

    print(
        "Classification Report"
    )

    print("-" * 70)

    print(
        get_classification_report(
            test_true,
            test_pred,
        )
    )


    # ========================================================
    # Confusion Matrix
    # ========================================================

    confusion_path = (
        CONFUSION_DIR
        / "bert.png"
    )

    save_confusion_matrix(
        test_true,
        test_pred,
        confusion_path,
    )


    # ========================================================
    # Save Tokenizer
    # ========================================================

    tokenizer_path = (
        MODEL_DIR
        / "bert_tokenizer"
    )

    tokenizer.save_pretrained(
        tokenizer_path
    )


    # ========================================================
    # Save Model
    # ========================================================

    model_path = (
        MODEL_DIR
        / "bert_model"
    )

    model.save_pretrained(
        model_path
    )


    # ========================================================
    # Save Metrics
    # ========================================================

    metrics_record = {

        "model":
            "BERT",

        "model_name":
            MODEL_NAME,

        "best_epoch":
            best_epoch,

        "best_val_macro_f1":
            best_macro_f1,

        "test_loss":
            test_loss,

        "accuracy":
            test_metrics[
                "accuracy"
            ],

        "macro_precision":
            test_metrics[
                "macro_precision"
            ],

        "macro_recall":
            test_metrics[
                "macro_recall"
            ],

        "macro_f1":
            test_metrics[
                "macro_f1"
            ],

        "weighted_precision":
            test_metrics[
                "weighted_precision"
            ],

        "weighted_recall":
            test_metrics[
                "weighted_recall"
            ],

        "weighted_f1":
            test_metrics[
                "weighted_f1"
            ],

        "training_time_sec":
            total_training_time,

        "inference_time_sec":
            inference_time,

        "total_parameters":
            total_params,

        "trainable_parameters":
            trainable_params,

        "max_length":
            MAX_LENGTH,

        "learning_rate":
            LEARNING_RATE,

        "batch_size":
            BATCH_SIZE,

        "gradient_accumulation_steps":
            GRADIENT_ACCUMULATION_STEPS,

        "epochs":
            EPOCHS,
    }


    metrics_path = (
        RESULTS_DIR
        / "bert_metrics.csv"
    )

    pd.DataFrame(
        [metrics_record]
    ).to_csv(
        metrics_path,
        index=False,
    )


    # ========================================================
    # Saved Artifacts
    # ========================================================

    print("=" * 70)
    print("Saved Artifacts")
    print("=" * 70)

    print(
        f"Checkpoint : "
        f"{best_model_path}"
    )

    print(
        f"Model      : "
        f"{model_path}"
    )

    print(
        f"Tokenizer  : "
        f"{tokenizer_path}"
    )

    print(
        f"Curve      : "
        f"{RESULTS_DIR / 'bert_training_curve.png'}"
    )

    print(
        f"F1 Curve   : "
        f"{RESULTS_DIR / 'bert_val_macro_f1.png'}"
    )

    print(
        f"Confusion  : "
        f"{confusion_path}"
    )

    print(
        f"Metrics    : "
        f"{metrics_path}"
    )

    print()


    print("=" * 70)
    print(
        "06-2. BERT Fine-tuning Completed"
    )
    print("=" * 70)


if __name__ == "__main__":
    main()

06-2. BERT Fine-tuning
Device                 : cuda
Model                  : google-bert/bert-base-uncased
Max Length             : 128
Batch Size             : 16
Gradient Accumulation  : 4
Effective Batch Size   : 64
Epochs                 : 3
Learning Rate          : 2e-05
Weight Decay           : 0.01
Warmup Ratio           : 0.1

Dataset
----------------------------------------------------------------------
Train      : 305,345
Validation : 65,422
Test       : 65,355

Tokenizer
----------------------------------------------------------------------
Tokenizer  : BertTokenizer
Vocab Size : 30,522

Tokenization Check
----------------------------------------------------------------------
Text  : im feeling rather rotten so im not very ambitious right now
Label : 0
Input IDs Shape : torch.Size([1, 13])
Attention Mask Shape : torch.Size([1, 13])
First Token ID : 101

Model
----------------------------------------------------------------------


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total Parameters     : 109,486,854
Trainable Parameters : 109,486,854

Training Schedule
----------------------------------------------------------------------
Train Batches         : 19,085
Optimizer Steps/Epoch : 4,772
Total Update Steps    : 14,316
Warmup Steps          : 1,431

Epoch [01/3]
Train Loss       : 0.2497
Val Loss         : 0.0878
Val Accuracy     : 0.9425
Val Macro F1     : 0.9028
Epoch Time       : 523.10s
Best model saved (Val Macro F1 = 0.9028)

Epoch [02/3]
Train Loss       : 0.0864
Val Loss         : 0.0878
Val Accuracy     : 0.9430
Val Macro F1     : 0.9179
Epoch Time       : 537.76s
Best model saved (Val Macro F1 = 0.9179)

Epoch [03/3]
Train Loss       : 0.0802
Val Loss         : 0.0885
Val Accuracy     : 0.9430
Val Macro F1     : 0.9047
Epoch Time       : 522.52s

Best Model
Best Epoch          : 2
Best Val Macro F1    : 0.9179

Final Test Evaluation

Test Loss         : 0.0892
Accuracy          : 0.9427
Macro Precision   : 0.8926
Macro Recall      : 0.9542
Mac

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Artifacts
Checkpoint : results/checkpoints/bert_best.pt
Model      : results/checkpoints/bert_model
Tokenizer  : results/checkpoints/bert_tokenizer
Curve      : results/bert_training_curve.png
F1 Curve   : results/bert_val_macro_f1.png
Confusion  : results/confusion_matrices/bert.png
Metrics    : results/bert_metrics.csv

06-2. BERT Fine-tuning Completed


In [3]:

import time

import matplotlib.pyplot as plt
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from utils.config import (
    SEED,
    NUM_CLASSES,
    PRETRAINED_MAX_LEN,
    PRETRAINED_LR,
    NUM_WORKERS,
    RESULTS_DIR,
    CONFUSION_DIR,
    MODEL_DIR,
    device,
    set_seed,
)

from utils.metrics import (
    calculate_metrics,
    get_classification_report,
    print_metrics,
)


# ============================================================
# 06-3. RoBERTa Baseline Fine-tuning
# ============================================================

MODEL_NAME = "FacebookAI/roberta-base"

BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 4
EPOCHS = 3

WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

NUM_LABELS = NUM_CLASSES
MAX_LENGTH = PRETRAINED_MAX_LEN
LEARNING_RATE = PRETRAINED_LR


# ============================================================
# Dataset
# ============================================================

class EmotionDataset(Dataset):

    def __init__(self, dataframe):
        self.texts = dataframe["text"].astype(str).tolist()
        self.labels = dataframe["label"].astype(int).tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return {
            "text": self.texts[index],
            "labels": self.labels[index],
        }


# ============================================================
# Dynamic Padding
# ============================================================

class RoBERTaCollator:

    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        texts = [item["text"] for item in batch]
        labels = [item["labels"] for item in batch]

        encoded = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        encoded["labels"] = torch.tensor(
            labels,
            dtype=torch.long,
        )

        return encoded


# ============================================================
# Utilities
# ============================================================

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )
    return total, trainable


def move_batch_to_device(batch):
    return {
        key: value.to(device)
        for key, value in batch.items()
    }


# ============================================================
# Training
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    scaler,
):
    model.train()
    total_loss = 0.0
    total_samples = 0

    optimizer.zero_grad(set_to_none=True)
    use_amp = device.type == "cuda"

    for step, batch in enumerate(dataloader):
        batch = move_batch_to_device(batch)

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss

        scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS

        if use_amp:
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()

        batch_size = batch["labels"].size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        update_step = (
            (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
            or step + 1 == len(dataloader)
        )

        if update_step:
            if use_amp:
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            if use_amp:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    return total_loss / total_samples


# ============================================================
# Evaluation
# ============================================================

def evaluate(model, dataloader):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    all_labels = []
    all_predictions = []

    use_amp = device.type == "cuda"

    with torch.no_grad():
        for batch in dataloader:
            batch = move_batch_to_device(batch)

            # Inference에서는 mixed precision을 사용해도
            # 되며, 평가 지표는 argmax 기준으로 계산한다.
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=use_amp,
            ):
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"],
                )

            batch_size = batch["labels"].size(0)
            total_loss += outputs.loss.item() * batch_size
            total_samples += batch_size

            predictions = outputs.logits.argmax(dim=1)

            all_labels.extend(
                batch["labels"].cpu().numpy()
            )
            all_predictions.extend(
                predictions.cpu().numpy()
            )

    average_loss = total_loss / total_samples
    metrics = calculate_metrics(all_labels, all_predictions)

    return average_loss, all_labels, all_predictions, metrics


# ============================================================
# Plotting
# ============================================================

def save_training_curve(history, save_path):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(10, 5))
    plt.plot(epochs, history["train_loss"], marker="o", label="Train Loss")
    plt.plot(epochs, history["val_loss"], marker="o", label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("RoBERTa Training Curve")
    plt.xticks(list(epochs))
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def save_f1_curve(history, save_path):
    epochs = range(1, len(history["val_macro_f1"]) + 1)

    plt.figure(figsize=(10, 5))
    plt.plot(
        epochs,
        history["val_macro_f1"],
        marker="o",
        label="Validation Macro F1",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.title("RoBERTa Validation Macro F1")
    plt.xticks(list(epochs))
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def save_confusion_matrix(y_true, y_pred, save_path):
    cm = confusion_matrix(y_true, y_pred)
    display = ConfusionMatrixDisplay(confusion_matrix=cm)
    display.plot(values_format="d")
    plt.title("RoBERTa Confusion Matrix")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


# ============================================================
# Main
# ============================================================

def main():
    set_seed(SEED)

    print("=" * 70)
    print("06-3. RoBERTa Fine-tuning")
    print("=" * 70)

    print(f"Device                 : {device}")
    print(f"Model                  : {MODEL_NAME}")
    print(f"Max Length             : {MAX_LENGTH}")
    print(f"Batch Size             : {BATCH_SIZE}")
    print(f"Gradient Accumulation  : {GRADIENT_ACCUMULATION_STEPS}")
    print(f"Effective Batch Size   : {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
    print(f"Epochs                 : {EPOCHS}")
    print(f"Learning Rate          : {LEARNING_RATE}")
    print(f"Weight Decay           : {WEIGHT_DECAY}")
    print(f"Warmup Ratio           : {WARMUP_RATIO}")
    print()

    # --------------------------------------------------------
    # Load existing splits
    # --------------------------------------------------------

    train_df = pd.read_csv("results/splits/train.csv")
    val_df = pd.read_csv("results/splits/validation.csv")
    test_df = pd.read_csv("results/splits/test.csv")

    print("Dataset")
    print("-" * 70)
    print(f"Train      : {len(train_df):,}")
    print(f"Validation : {len(val_df):,}")
    print(f"Test       : {len(test_df):,}")
    print()

    # --------------------------------------------------------
    # Tokenizer and DataLoaders
    # --------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    print("Tokenizer")
    print("-" * 70)
    print(f"Tokenizer  : {tokenizer.__class__.__name__}")
    print(f"Vocab Size : {tokenizer.vocab_size:,}")
    print()

    collator = RoBERTaCollator(tokenizer, MAX_LENGTH)

    train_loader = DataLoader(
        EmotionDataset(train_df),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )

    val_loader = DataLoader(
        EmotionDataset(val_df),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )

    test_loader = DataLoader(
        EmotionDataset(test_df),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
    )

    sample = tokenizer(
        str(train_df.iloc[0]["text"]),
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    print("Tokenization Check")
    print("-" * 70)
    print(f"Text : {train_df.iloc[0]['text']}")
    print(f"Label : {train_df.iloc[0]['label']}")
    print(f"Input IDs Shape : {sample['input_ids'].shape}")
    print(f"Attention Mask Shape : {sample['attention_mask'].shape}")
    print(f"First Token ID : {sample['input_ids'][0, 0].item()}")
    print()

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    id2label = {i: f"LABEL_{i}" for i in range(NUM_LABELS)}
    label2id = {v: k for k, v in id2label.items()}

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=id2label,
        label2id=label2id,
    ).to(device)

    total_params, trainable_params = count_parameters(model)

    print("Model")
    print("-" * 70)
    print(f"Total Parameters     : {total_params:,}")
    print(f"Trainable Parameters : {trainable_params:,}")
    print()

    # --------------------------------------------------------
    # Optimizer and scheduler
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    updates_per_epoch = (
        len(train_loader) + GRADIENT_ACCUMULATION_STEPS - 1
    ) // GRADIENT_ACCUMULATION_STEPS

    total_update_steps = updates_per_epoch * EPOCHS
    warmup_steps = int(total_update_steps * WARMUP_RATIO)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_update_steps,
    )

    print("Training Schedule")
    print("-" * 70)
    print(f"Train Batches         : {len(train_loader):,}")
    print(f"Optimizer Steps/Epoch : {updates_per_epoch:,}")
    print(f"Total Update Steps    : {total_update_steps:,}")
    print(f"Warmup Steps          : {warmup_steps:,}")
    print()

    # --------------------------------------------------------
    # Training loop
    # --------------------------------------------------------

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_macro_f1": [],
    }

    best_macro_f1 = -1.0
    best_epoch = -1
    best_model_path = MODEL_DIR / "roberta_best.pt"

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda"),
    )

    training_start = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.perf_counter()

        print("=" * 70)
        print(f"Epoch [{epoch:02d}/{EPOCHS}]")
        print("=" * 70)

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            scaler,
        )

        val_loss, val_true, val_pred, val_metrics = evaluate(
            model,
            val_loader,
        )

        val_macro_f1 = val_metrics["macro_f1"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_macro_f1"].append(val_macro_f1)

        epoch_time = time.perf_counter() - epoch_start

        print(f"Train Loss       : {train_loss:.4f}")
        print(f"Val Loss         : {val_loss:.4f}")
        print(f"Val Accuracy     : {val_metrics['accuracy']:.4f}")
        print(f"Val Macro F1     : {val_macro_f1:.4f}")
        print(f"Epoch Time       : {epoch_time:.2f}s")

        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            best_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "best_macro_f1": best_macro_f1,
                    "model_name": MODEL_NAME,
                    "max_length": MAX_LENGTH,
                    "learning_rate": LEARNING_RATE,
                    "batch_size": BATCH_SIZE,
                    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
                },
                best_model_path,
            )

            print(
                f"Best model saved "
                f"(Val Macro F1 = {best_macro_f1:.4f})"
            )

        print()

    total_training_time = time.perf_counter() - training_start

    # --------------------------------------------------------
    # Save training curves
    # --------------------------------------------------------

    save_training_curve(
        history,
        RESULTS_DIR / "roberta_training_curve.png",
    )

    save_f1_curve(
        history,
        RESULTS_DIR / "roberta_val_macro_f1.png",
    )

    # --------------------------------------------------------
    # Restore best checkpoint
    # --------------------------------------------------------

    checkpoint = torch.load(
        best_model_path,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])

    print("=" * 70)
    print("Best Model")
    print("=" * 70)
    print(f"Best Epoch          : {checkpoint['epoch']}")
    print(f"Best Val Macro F1    : {checkpoint['best_macro_f1']:.4f}")
    print()

    # --------------------------------------------------------
    # Final test evaluation: run only after model selection
    # --------------------------------------------------------

    print("=" * 70)
    print("Final Test Evaluation")
    print("=" * 70)

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_start = time.perf_counter()

    test_loss, test_true, test_pred, test_metrics = evaluate(
        model,
        test_loader,
    )

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time = time.perf_counter() - inference_start

    print(f"Test Loss         : {test_loss:.4f}")
    print_metrics(test_metrics)
    print(f"Inference Time    : {inference_time:.2f}s")
    print()

    print("Classification Report")
    print("-" * 70)
    print(get_classification_report(test_true, test_pred))

    # --------------------------------------------------------
    # Save artifacts
    # --------------------------------------------------------

    confusion_path = CONFUSION_DIR / "roberta.png"
    save_confusion_matrix(test_true, test_pred, confusion_path)

    tokenizer_path = MODEL_DIR / "roberta_tokenizer"
    tokenizer.save_pretrained(tokenizer_path)

    model_path = MODEL_DIR / "roberta_model"
    model.save_pretrained(model_path)

    metrics_record = {
        "model": "RoBERTa",
        "model_name": MODEL_NAME,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_macro_f1,
        "test_loss": test_loss,
        **test_metrics,
        "training_time_sec": total_training_time,
        "inference_time_sec": inference_time,
        "total_parameters": total_params,
        "trainable_parameters": trainable_params,
        "max_length": MAX_LENGTH,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "epochs": EPOCHS,
    }

    metrics_path = RESULTS_DIR / "roberta_metrics.csv"

    pd.DataFrame([metrics_record]).to_csv(
        metrics_path,
        index=False,
    )

    print("=" * 70)
    print("Saved Artifacts")
    print("=" * 70)
    print(f"Checkpoint : {best_model_path}")
    print(f"Model      : {model_path}")
    print(f"Tokenizer  : {tokenizer_path}")
    print(f"Curve      : {RESULTS_DIR / 'roberta_training_curve.png'}")
    print(f"F1 Curve   : {RESULTS_DIR / 'roberta_val_macro_f1.png'}")
    print(f"Confusion  : {confusion_path}")
    print(f"Metrics    : {metrics_path}")
    print()

    print("=" * 70)
    print("06-3. RoBERTa Fine-tuning Completed")
    print("=" * 70)


if __name__ == "__main__":
    main()

06-3. RoBERTa Fine-tuning
Device                 : cuda
Model                  : FacebookAI/roberta-base
Max Length             : 128
Batch Size             : 16
Gradient Accumulation  : 4
Effective Batch Size   : 64
Epochs                 : 3
Learning Rate          : 2e-05
Weight Decay           : 0.01
Warmup Ratio           : 0.1

Dataset
----------------------------------------------------------------------
Train      : 305,345
Validation : 65,422
Test       : 65,355

Tokenizer
----------------------------------------------------------------------
Tokenizer  : RobertaTokenizer
Vocab Size : 50,265

Tokenization Check
----------------------------------------------------------------------
Text : im feeling rather rotten so im not very ambitious right now
Label : 0
Input IDs Shape : torch.Size([1, 13])
Attention Mask Shape : torch.Size([1, 13])
First Token ID : 0



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model
----------------------------------------------------------------------
Total Parameters     : 124,650,246
Trainable Parameters : 124,650,246

Training Schedule
----------------------------------------------------------------------
Train Batches         : 19,085
Optimizer Steps/Epoch : 4,772
Total Update Steps    : 14,316
Warmup Steps          : 1,431

Epoch [01/3]
Train Loss       : 0.2492
Val Loss         : 0.0987
Val Accuracy     : 0.9420
Val Macro F1     : 0.9100
Epoch Time       : 515.17s
Best model saved (Val Macro F1 = 0.9100)

Epoch [02/3]
Train Loss       : 0.0909
Val Loss         : 0.0912
Val Accuracy     : 0.9422
Val Macro F1     : 0.9046
Epoch Time       : 509.22s

Epoch [03/3]
Train Loss       : 0.0833
Val Loss         : 0.0899
Val Accuracy     : 0.9430
Val Macro F1     : 0.9020
Epoch Time       : 506.25s

Best Model
Best Epoch          : 1
Best Val Macro F1    : 0.9100

Final Test Evaluation
Test Loss         : 0.0971
Accuracy          : 0.9421
Macro Precision   : 0.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Artifacts
Checkpoint : results/checkpoints/roberta_best.pt
Model      : results/checkpoints/roberta_model
Tokenizer  : results/checkpoints/roberta_tokenizer
Curve      : results/roberta_training_curve.png
F1 Curve   : results/roberta_val_macro_f1.png
Confusion  : results/confusion_matrices/roberta.png
Metrics    : results/roberta_metrics.csv

06-3. RoBERTa Fine-tuning Completed


In [5]:

# ============================================================
# 06-4. DeBERTa-v3 Fine-tuning
# - Dynamic Padding
# - BF16 AMP (supported GPU) / FP32 fallback
# - Gradient Accumulation
# - Periodic Resumable Checkpoint
# - Validation Macro F1 based model selection
# - Final test evaluation
# ============================================================

import os
import time
import random
import shutil
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


# ============================================================
# 1. Configuration
# ============================================================

SEED = 42

MODEL_NAME = "microsoft/deberta-v3-base"
MODEL_KEY = "deberta"

NUM_CLASSES = 6
MAX_LEN = 128

BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS

EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

NUM_WORKERS = 0

# Progress log frequency
LOG_EVERY_BATCHES = 100

# Save a resumable checkpoint approximately every 100 batches.
# Checkpointing occurs only after an optimizer update.
SAVE_EVERY_BATCHES = 100

# True: resume from the existing latest checkpoint if available.
# False: start a fresh run and delete previous DeBERTa checkpoints.
RESUME = True

DATA_DIR = Path("results/splits")
RESULTS_DIR = Path("results")

CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
CONFUSION_DIR = RESULTS_DIR / "confusion_matrices"

TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "validation.csv"
TEST_PATH = DATA_DIR / "test.csv"

LAST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR / "deberta_last_checkpoint.pt"
)
BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR / "deberta_best.pt"
)

MODEL_SAVE_DIR = CHECKPOINT_DIR / "deberta_model"
TOKENIZER_SAVE_DIR = CHECKPOINT_DIR / "deberta_tokenizer"

HISTORY_PATH = RESULTS_DIR / "deberta_training_history.csv"
METRICS_PATH = RESULTS_DIR / "deberta_metrics.csv"
CLASS_REPORT_PATH = RESULTS_DIR / "deberta_classification_report.csv"

LOSS_PLOT_PATH = RESULTS_DIR / "deberta_training_curve.png"
F1_PLOT_PATH = RESULTS_DIR / "deberta_val_macro_f1.png"
CONFUSION_PLOT_PATH = CONFUSION_DIR / "deberta.png"

INTERRUPT_LOG_PATH = RESULTS_DIR / "deberta_interrupted.txt"

for path in [CHECKPOINT_DIR, RESULTS_DIR, CONFUSION_DIR]:
    path.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Reproducibility
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_BF16 = (
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

print("=" * 60)
print("DeBERTa-v3 Fine-tuning")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")
print(f"BF16 AMP: {USE_BF16}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Gradient accumulation: {GRAD_ACCUM_STEPS}")
print(f"Effective batch size: {EFFECTIVE_BATCH_SIZE}")
print(f"Max length: {MAX_LEN}")
print(f"Epochs: {EPOCHS}")
print(f"Learning rate: {LEARNING_RATE}")
print("=" * 60)

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        "GPU memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )


def amp_context():
    """Use BF16 autocast when supported; otherwise use FP32."""
    if USE_BF16:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16,
        )

    return nullcontext()


def synchronize_device():
    if device.type == "cuda":
        torch.cuda.synchronize()


# ============================================================
# 3. Data loading
# ============================================================

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Split file not found: {path.resolve()}"
        )


train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

required_columns = {"text", "label"}

for split_name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{split_name} CSV is missing columns: {missing_columns}"
        )

    if df["text"].isna().any():
        raise ValueError(
            f"{split_name} contains missing text values."
        )

    if df["label"].isna().any():
        raise ValueError(
            f"{split_name} contains missing labels."
        )

    df["text"] = df["text"].astype(str)
    df["label"] = df["label"].astype(int)

    if not df["label"].between(0, NUM_CLASSES - 1).all():
        raise ValueError(
            f"{split_name} has labels outside 0..{NUM_CLASSES - 1}"
        )

print("\nDataset sizes")
print(f"Train:      {len(train_df):,}")
print(f"Validation: {len(val_df):,}")
print(f"Test:       {len(test_df):,}")


# ============================================================
# 4. Tokenizer and Dataset
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)


class EmotionDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {
            "text": self.texts[idx],
            "label": self.labels[idx],
        }


def make_collate_fn(tokenizer):
    """
    Tokenize each batch with dynamic padding.
    Padding length is based on the longest sequence in that batch.
    """

    def collate_fn(batch):
        texts = [item["text"] for item in batch]
        labels = torch.tensor(
            [item["label"] for item in batch],
            dtype=torch.long,
        )

        inputs = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        )

        inputs["labels"] = labels
        return inputs

    return collate_fn


collate_fn = make_collate_fn(tokenizer)

train_dataset = EmotionDataset(train_df)
val_dataset = EmotionDataset(val_df)
test_dataset = EmotionDataset(test_df)

# The training generator is seeded separately for each epoch.
# This allows us to recreate the same shuffled batch order on resume.
def make_train_loader(epoch):
    generator = torch.Generator()
    generator.manual_seed(SEED + epoch)

    return DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),
        collate_fn=collate_fn,
        generator=generator,
    )


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    collate_fn=collate_fn,
)


# ============================================================
# 5. Model
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    torch_dtype=torch.float32,
)

model.to(device)
model.float()

# This implementation uses BF16 autocast, not FP16 GradScaler.
# Model parameters remain FP32.
non_fp32_params = [
    name
    for name, param in model.named_parameters()
    if param.requires_grad and param.dtype != torch.float32
]

if non_fp32_params:
    raise RuntimeError(
        "Expected FP32 trainable parameters, but found non-FP32 parameters."
    )

total_params = sum(
    param.numel() for param in model.parameters()
)

trainable_params = sum(
    param.numel()
    for param in model.parameters()
    if param.requires_grad
)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


# ============================================================
# 6. Optimizer and Scheduler
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

batches_per_epoch = len(train_loader) if "train_loader" in globals() else (
    (len(train_dataset) + BATCH_SIZE - 1) // BATCH_SIZE
)

optimizer_steps_per_epoch = (
    batches_per_epoch + GRAD_ACCUM_STEPS - 1
) // GRAD_ACCUM_STEPS

total_training_steps = optimizer_steps_per_epoch * EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

print(f"Batches per epoch: {batches_per_epoch:,}")
print(f"Optimizer steps per epoch: {optimizer_steps_per_epoch:,}")
print(f"Total optimizer steps: {total_training_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")


# ============================================================
# 7. Metrics and evaluation
# ============================================================

def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "weighted_precision": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "weighted_recall": recall_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
    }


@torch.inference_mode()
def evaluate(model, data_loader, measure_inference_time=False):
    model.eval()

    all_labels = []
    all_predictions = []

    total_loss = 0.0
    total_samples = 0

    synchronize_device()
    start_time = time.perf_counter()

    for batch in data_loader:
        labels = batch.pop("labels").to(device, non_blocking=True)

        inputs = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        with amp_context():
            outputs = model(**inputs, labels=labels)

        batch_size = labels.size(0)
        total_loss += outputs.loss.item() * batch_size
        total_samples += batch_size

        predictions = outputs.logits.argmax(dim=-1)

        all_labels.extend(labels.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())

    synchronize_device()
    elapsed_time = time.perf_counter() - start_time

    metrics = calculate_metrics(
        all_labels,
        all_predictions,
    )

    metrics["loss"] = total_loss / max(total_samples, 1)

    if measure_inference_time:
        metrics["inference_seconds"] = elapsed_time
        metrics["inference_samples_per_second"] = (
            total_samples / elapsed_time if elapsed_time > 0 else 0
        )

    return metrics, all_labels, all_predictions


# ============================================================
# 8. Checkpoint utilities
# ============================================================

def capture_rng_state():
    state = {
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
    }

    if torch.cuda.is_available():
        state["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()

    return state


def restore_rng_state(state):
    random.setstate(state["python_rng_state"])
    np.random.set_state(state["numpy_rng_state"])
    torch.set_rng_state(state["torch_rng_state"])

    if (
        torch.cuda.is_available()
        and "cuda_rng_state_all" in state
    ):
        torch.cuda.set_rng_state_all(
            state["cuda_rng_state_all"]
        )


def atomic_torch_save(obj, path):
    """
    Write to a temporary file first, then replace the destination.
    This reduces the risk of leaving a partially written checkpoint.
    """
    path = Path(path)
    temp_path = path.with_suffix(path.suffix + ".tmp")

    torch.save(obj, temp_path)
    os.replace(temp_path, path)


def save_last_checkpoint(
    next_epoch,
    next_batch_idx,
    global_step,
    best_epoch,
    best_val_macro_f1,
    history,
):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "next_epoch": next_epoch,
        "next_batch_idx": next_batch_idx,
        "global_step": global_step,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "history": history,
        "rng_state": capture_rng_state(),
        "model_name": MODEL_NAME,
        "seed": SEED,
        "max_len": MAX_LEN,
        "num_classes": NUM_CLASSES,
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
    }

    atomic_torch_save(
        checkpoint,
        LAST_CHECKPOINT_PATH,
    )


def save_best_checkpoint(epoch, val_macro_f1):
    best_state = {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }

    atomic_torch_save(
        {
            "model_state_dict": best_state,
            "best_epoch": epoch + 1,
            "best_val_macro_f1": val_macro_f1,
            "model_name": MODEL_NAME,
            "num_classes": NUM_CLASSES,
            "max_len": MAX_LEN,
        },
        BEST_CHECKPOINT_PATH,
    )


# ============================================================
# 9. Initialize or resume
# ============================================================

if not RESUME:
    for path in [
        LAST_CHECKPOINT_PATH,
        BEST_CHECKPOINT_PATH,
    ]:
        if path.exists():
            path.unlink()

    print("\nStarting a fresh training run.")

history = []
global_step = 0
best_epoch = 0
best_val_macro_f1 = -1.0

start_epoch = 0
start_batch_idx = 0
resume_rng_state = None

if RESUME and LAST_CHECKPOINT_PATH.exists():
    print("\nResuming from the latest checkpoint...")

    # This checkpoint is a local file created by this script.
    # weights_only=False is needed because it contains RNG/history metadata.
    checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=device,
        weights_only=False,
    )

    if checkpoint["model_name"] != MODEL_NAME:
        raise ValueError("Checkpoint model name does not match.")

    if checkpoint["seed"] != SEED:
        raise ValueError("Checkpoint seed does not match.")

    if checkpoint["max_len"] != MAX_LEN:
        raise ValueError("Checkpoint MAX_LEN does not match.")

    if checkpoint["num_classes"] != NUM_CLASSES:
        raise ValueError("Checkpoint NUM_CLASSES does not match.")

    if checkpoint["batch_size"] != BATCH_SIZE:
        raise ValueError("Checkpoint batch size does not match.")

    if checkpoint["grad_accum_steps"] != GRAD_ACCUM_STEPS:
        raise ValueError("Checkpoint accumulation steps do not match.")

    if checkpoint["epochs"] != EPOCHS:
        raise ValueError("Checkpoint EPOCHS does not match.")

    if checkpoint["learning_rate"] != LEARNING_RATE:
        raise ValueError("Checkpoint learning rate does not match.")

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint["next_epoch"]
    start_batch_idx = checkpoint["next_batch_idx"]
    global_step = checkpoint["global_step"]
    best_epoch = checkpoint["best_epoch"]
    best_val_macro_f1 = checkpoint["best_val_macro_f1"]
    history = checkpoint["history"]
    resume_rng_state = checkpoint["rng_state"]

    print(f"Resume epoch: {start_epoch + 1}")
    print(f"Resume batch index: {start_batch_idx}")
    print(f"Global optimizer step: {global_step}")
    print(f"Best epoch so far: {best_epoch}")
    print(f"Best validation Macro F1: {best_val_macro_f1:.4f}")

    if start_epoch >= EPOCHS:
        print(
            "The checkpoint indicates that all epochs are complete. "
            "If this was intentional, proceed to final evaluation."
        )

elif RESUME:
    print("\nNo resumable checkpoint found. Starting from scratch.")


# ============================================================
# 10. Training
# ============================================================

training_start_time = time.perf_counter()

try:
    for epoch in range(start_epoch, EPOCHS):
        train_loader = make_train_loader(epoch)

        # On resume, recreate the deterministic epoch ordering and skip
        # batches that were already processed before the checkpoint.
        batch_start = (
            start_batch_idx if epoch == start_epoch else 0
        )

        if batch_start > len(train_loader):
            raise ValueError(
                "Checkpoint batch index exceeds the current epoch length."
            )

        model.train()
        optimizer.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_sample_count = 0

        epoch_start_time = time.perf_counter()
        last_log_time = epoch_start_time

        train_iterator = iter(train_loader)

        for _ in range(batch_start):
            next(train_iterator)

        # Restore the saved random state after rebuilding/skipping the
        # DataLoader iterator so dropout continues from the saved state.
        if (
            resume_rng_state is not None
            and epoch == start_epoch
        ):
            restore_rng_state(resume_rng_state)
            resume_rng_state = None

        print(f"\n{'=' * 60}")
        print(f"Epoch {epoch + 1}/{EPOCHS}")
        print(f"Resuming at batch {batch_start + 1:,}")
        print(f"{'=' * 60}")

        for batch_idx in range(batch_start, len(train_loader)):
            batch = next(train_iterator)

            labels = batch.pop("labels").to(
                device,
                non_blocking=True,
            )

            inputs = {
                key: value.to(device, non_blocking=True)
                for key, value in batch.items()
            }

            # Size of the current accumulation group.
            group_start = (
                batch_idx // GRAD_ACCUM_STEPS
            ) * GRAD_ACCUM_STEPS

            group_size = min(
                GRAD_ACCUM_STEPS,
                len(train_loader) - group_start,
            )

            with amp_context():
                outputs = model(**inputs, labels=labels)
                loss = outputs.loss

            # Divide by the actual group size, including the final
            # incomplete accumulation group.
            (loss / group_size).backward()

            batch_size = labels.size(0)
            epoch_loss_sum += loss.item() * batch_size
            epoch_sample_count += batch_size

            is_accumulation_boundary = (
                (batch_idx + 1) % GRAD_ACCUM_STEPS == 0
            )

            is_last_batch = (
                batch_idx + 1 == len(train_loader)
            )

            if is_accumulation_boundary or is_last_batch:
                nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                )

                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

                global_step += 1

                completed_batches = batch_idx + 1

                # Save only after optimizer updates, so a checkpoint
                # never requires restoring partially accumulated gradients.
                if (
                    completed_batches % SAVE_EVERY_BATCHES == 0
                    or is_last_batch
                ):
                    next_epoch = epoch
                    next_batch_idx = completed_batches

                    if is_last_batch:
                        next_epoch = epoch + 1
                        next_batch_idx = 0

                    save_last_checkpoint(
                        next_epoch=next_epoch,
                        next_batch_idx=next_batch_idx,
                        global_step=global_step,
                        best_epoch=best_epoch,
                        best_val_macro_f1=best_val_macro_f1,
                        history=history,
                    )

            completed_batches = batch_idx + 1

            if (
                completed_batches % LOG_EVERY_BATCHES == 0
                or completed_batches == len(train_loader)
            ):
                now = time.perf_counter()
                interval = now - last_log_time
                last_log_time = now

                progress_pct = (
                    completed_batches / len(train_loader) * 100
                )

                avg_loss = (
                    epoch_loss_sum / max(epoch_sample_count, 1)
                )

                print(
                    f"Epoch {epoch + 1}/{EPOCHS} | "
                    f"Batch {completed_batches:,}/{len(train_loader):,} "
                    f"({progress_pct:.1f}%) | "
                    f"Train Loss: {avg_loss:.4f} | "
                    f"Interval: {interval:.1f}s | "
                    f"Optimizer Step: {global_step:,}"
                )

        # --------------------------------------------------------
        # Validation at the end of each epoch
        # --------------------------------------------------------

        train_loss = (
            epoch_loss_sum / max(epoch_sample_count, 1)
        )

        val_metrics, _, _ = evaluate(
            model,
            val_loader,
        )

        epoch_seconds = time.perf_counter() - epoch_start_time

        epoch_record = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_precision": val_metrics["macro_precision"],
            "val_macro_recall": val_metrics["macro_recall"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "epoch_seconds": epoch_seconds,
            "global_step": global_step,
        }

        history.append(epoch_record)

        print(f"\nEpoch {epoch + 1} summary")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Validation Loss: {val_metrics['loss']:.4f}")
        print(f"Validation Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"Validation Macro F1: {val_metrics['macro_f1']:.4f}")
        print(f"Validation Weighted F1: {val_metrics['weighted_f1']:.4f}")
        print(f"Epoch time: {epoch_seconds / 60:.2f} min")

        # Select the best checkpoint using validation Macro F1 only.
        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch + 1

            save_best_checkpoint(
                epoch=epoch,
                val_macro_f1=best_val_macro_f1,
            )

            print(
                f"Best checkpoint updated: "
                f"epoch={best_epoch}, "
                f"val_macro_f1={best_val_macro_f1:.4f}"
            )

        # Persist history and the epoch-boundary checkpoint.
        pd.DataFrame(history).to_csv(
            HISTORY_PATH,
            index=False,
        )

        save_last_checkpoint(
            next_epoch=epoch + 1,
            next_batch_idx=0,
            global_step=global_step,
            best_epoch=best_epoch,
            best_val_macro_f1=best_val_macro_f1,
            history=history,
        )

        # The next epoch starts from batch zero.
        start_batch_idx = 0

    total_training_seconds = (
        time.perf_counter() - training_start_time
    )

    print("\nTraining completed.")
    print(f"Best epoch: {best_epoch}")
    print(f"Best validation Macro F1: {best_val_macro_f1:.4f}")
    print(
        f"Training time this session: "
        f"{total_training_seconds / 60:.2f} min"
    )

except KeyboardInterrupt:
    # Keep the last atomically saved checkpoint.
    # Do not overwrite it with potentially incomplete gradients.
    message = (
        f"Training interrupted at {time.strftime('%Y-%m-%d %H:%M:%S')}\n"
        f"Latest saved checkpoint: {LAST_CHECKPOINT_PATH}\n"
        f"Best model checkpoint: {BEST_CHECKPOINT_PATH}\n"
        f"Resume by rerunning this script with RESUME=True.\n"
    )

    INTERRUPT_LOG_PATH.write_text(
        message,
        encoding="utf-8",
    )

    print("\n\nTraining interrupted.")
    print(message)
    print(
        "Rerun the script with RESUME=True to continue from "
        "the latest saved checkpoint."
    )

    raise


# ============================================================
# 11. Final evaluation
# ============================================================

# This section runs only if training finishes normally.
# Test data is not used for checkpoint selection.

if not LAST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "Latest checkpoint not found after training."
    )

last_checkpoint = torch.load(
    LAST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

if last_checkpoint["next_epoch"] < EPOCHS:
    raise RuntimeError(
        "Training is not complete. Resume training before test evaluation."
    )

if not BEST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "Best checkpoint not found. Validation selection did not complete."
    )

best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=True,
)

model.load_state_dict(best_checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("\n" + "=" * 60)
print("Final Test Evaluation")
print("=" * 60)
print(f"Selected epoch: {best_checkpoint['best_epoch']}")
print(
    "Best validation Macro F1: "
    f"{best_checkpoint['best_val_macro_f1']:.4f}"
)

test_metrics, y_true, y_pred = evaluate(
    model,
    test_loader,
    measure_inference_time=True,
)

test_metrics["model"] = MODEL_KEY
test_metrics["best_epoch"] = best_checkpoint["best_epoch"]
test_metrics["best_val_macro_f1"] = (
    best_checkpoint["best_val_macro_f1"]
)
test_metrics["total_params"] = total_params
test_metrics["trainable_params"] = trainable_params
test_metrics["training_seconds_this_session"] = (
    time.perf_counter() - training_start_time
)

print("\nTest metrics")
for key, value in test_metrics.items():
    if isinstance(value, (int, float, np.integer, np.floating)):
        print(f"{key}: {value:.4f}" if isinstance(value, (float, np.floating)) else f"{key}: {value}")
    else:
        print(f"{key}: {value}")

pd.DataFrame([test_metrics]).to_csv(
    METRICS_PATH,
    index=False,
)

# Detailed per-class classification report
class_report = classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    digits=4,
    zero_division=0,
    output_dict=True,
)

pd.DataFrame(class_report).transpose().to_csv(
    CLASS_REPORT_PATH,
    index=True,
)

print("\nClassification report")
print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        digits=4,
        zero_division=0,
    )
)


# ============================================================
# 12. Save final model and tokenizer
# ============================================================

model.save_pretrained(MODEL_SAVE_DIR)
tokenizer.save_pretrained(TOKENIZER_SAVE_DIR)

print(f"\nModel saved: {MODEL_SAVE_DIR}")
print(f"Tokenizer saved: {TOKENIZER_SAVE_DIR}")


# ============================================================
# 13. Save training curves
# ============================================================

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

if not history_df.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(
        history_df["epoch"],
        history_df["train_loss"],
        marker="o",
        label="Train Loss",
    )
    plt.plot(
        history_df["epoch"],
        history_df["val_loss"],
        marker="o",
        label="Validation Loss",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("DeBERTa-v3 Training and Validation Loss")
    plt.xticks(history_df["epoch"])
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(LOSS_PLOT_PATH, dpi=200)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(
        history_df["epoch"],
        history_df["val_macro_f1"],
        marker="o",
        label="Validation Macro F1",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.title("DeBERTa-v3 Validation Macro F1")
    plt.xticks(history_df["epoch"])
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(F1_PLOT_PATH, dpi=200)
    plt.close()


# ============================================================
# 14. Save confusion matrix
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(range(NUM_CLASSES)),
)

fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False,
)
ax.set_title("DeBERTa-v3 Confusion Matrix")
fig.tight_layout()
fig.savefig(CONFUSION_PLOT_PATH, dpi=200)
plt.close(fig)


# ============================================================
# 15. Final summary
# ============================================================

print("\n" + "=" * 60)
print("DeBERTa-v3 Experiment Finished")
print("=" * 60)
print(f"Best epoch: {best_checkpoint['best_epoch']}")
print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test Macro Precision: {test_metrics['macro_precision']:.4f}")
print(f"Test Macro Recall: {test_metrics['macro_recall']:.4f}")
print(f"Test Macro F1: {test_metrics['macro_f1']:.4f}")
print(f"Test Weighted F1: {test_metrics['weighted_f1']:.4f}")
print(f"Parameters: {total_params:,}")
print(
    f"Inference time: "
    f"{test_metrics['inference_seconds']:.2f} seconds"
)

print("\nSaved artifacts:")
for path in [
    BEST_CHECKPOINT_PATH,
    LAST_CHECKPOINT_PATH,
    MODEL_SAVE_DIR,
    TOKENIZER_SAVE_DIR,
    HISTORY_PATH,
    METRICS_PATH,
    CLASS_REPORT_PATH,
    LOSS_PLOT_PATH,
    F1_PLOT_PATH,
    CONFUSION_PLOT_PATH,
]:
    print(f"- {path}")

DeBERTa-v3 Fine-tuning
Model: microsoft/deberta-v3-base
Device: cuda
BF16 AMP: True
Batch size: 16
Gradient accumulation: 4
Effective batch size: 64
Max length: 128
Epochs: 3
Learning rate: 2e-05
GPU: NVIDIA GeForce RTX 5070 Ti
GPU memory: 15.92 GB

Dataset sizes
Train:      305,345
Validation: 65,422
Test:       65,355


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den


Total parameters: 184,426,758
Trainable parameters: 184,426,758
Batches per epoch: 19,085
Optimizer steps per epoch: 4,772
Total optimizer steps: 14,316
Warmup steps: 1,431

No resumable checkpoint found. Starting from scratch.

Epoch 1/3
Resuming at batch 1
Epoch 1/3 | Batch 100/19,085 (0.5%) | Train Loss: 1.8072 | Interval: 41.4s | Optimizer Step: 25
Epoch 1/3 | Batch 200/19,085 (1.0%) | Train Loss: 1.8027 | Interval: 32.3s | Optimizer Step: 50
Epoch 1/3 | Batch 300/19,085 (1.6%) | Train Loss: 1.7968 | Interval: 38.2s | Optimizer Step: 75
Epoch 1/3 | Batch 400/19,085 (2.1%) | Train Loss: 1.7880 | Interval: 36.2s | Optimizer Step: 100
Epoch 1/3 | Batch 500/19,085 (2.6%) | Train Loss: 1.7734 | Interval: 39.7s | Optimizer Step: 125
Epoch 1/3 | Batch 600/19,085 (3.1%) | Train Loss: 1.7533 | Interval: 35.0s | Optimizer Step: 150
Epoch 1/3 | Batch 700/19,085 (3.7%) | Train Loss: 1.7271 | Interval: 36.4s | Optimizer Step: 175
Epoch 1/3 | Batch 800/19,085 (4.2%) | Train Loss: 1.7100 | Inter

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved: results/checkpoints/deberta_model
Tokenizer saved: results/checkpoints/deberta_tokenizer

DeBERTa-v3 Experiment Finished
Best epoch: 3
Test Accuracy: 0.9431
Test Macro Precision: 0.9216
Test Macro Recall: 0.9102
Test Macro F1: 0.9081
Test Weighted F1: 0.9432
Parameters: 184,426,758
Inference time: 51.43 seconds

Saved artifacts:
- results/checkpoints/deberta_best.pt
- results/checkpoints/deberta_last_checkpoint.pt
- results/checkpoints/deberta_model
- results/checkpoints/deberta_tokenizer
- results/deberta_training_history.csv
- results/deberta_metrics.csv
- results/deberta_classification_report.csv
- results/deberta_training_curve.png
- results/deberta_val_macro_f1.png
- results/confusion_matrices/deberta.png
